# 深度学习实验:扩散模型 (DDPM) — 代码模板

**学号:** ___________  **姓名:** ___________

> 提交前请先 `Kernel → Restart & Run All`,确保所有 cell 顺序运行无报错。  
> 所有需要你补全的代码位置都用 `# TODO:` 标出;思考题写在对应 Markdown cell 中。


## 0. 环境与依赖

本实验仅需 CPU。如安装缺失,请先运行:

```
pip install torch torchvision numpy matplotlib tqdm scikit-learn
```


In [ ]:
import math
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

# 固定随机种子,保证可复现
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cpu")   # 本实验全程使用 CPU
print("PyTorch version:", torch.__version__)
print("Device:", device)


---
## Task 1:前向扩散过程的实现与可视化  (20 分)

### 数学回顾

设 $\beta_t$ 为线性噪声调度,$\alpha_t = 1-\beta_t$,$\bar\alpha_t = \prod_{s=1}^t \alpha_s$,则:

$$
x_t = \sqrt{\bar\alpha_t}\,x_0 + \sqrt{1-\bar\alpha_t}\,\epsilon,\quad \epsilon\sim\mathcal N(0,I)
$$

请按下方提示完成实现。


### 1.1 噪声调度与累积系数(TODO)

实现:
- `make_beta_schedule(T, beta_start=1e-4, beta_end=0.02)`:返回长度为 `T` 的线性 $\beta$ 数组(`torch.Tensor`)。
- 由 `betas` 推导 `alphas` 和 `alphas_cumprod`($\bar\alpha_t$)。


In [ ]:
def make_beta_schedule(T: int, beta_start: float = 1e-4, beta_end: float = 0.02) -> torch.Tensor:
    '''线性 beta 调度,返回形状 (T,) 的 1D tensor。'''
    # TODO: 用 torch.linspace 构造 β_1, ..., β_T
    raise NotImplementedError


T = 1000
betas = make_beta_schedule(T)
assert betas.shape == (T,)
assert torch.isclose(betas[0], torch.tensor(1e-4))
assert torch.isclose(betas[-1], torch.tensor(0.02))

# TODO: 计算 alphas 与 alphas_cumprod (\bar\alpha_t)
alphas = None              # shape (T,)
alphas_cumprod = None      # shape (T,), 即 \bar\alpha_t

# Sanity check
# assert torch.isclose(alphas_cumprod[0], 1.0 - betas[0])
# assert (alphas_cumprod[1:] <= alphas_cumprod[:-1] + 1e-7).all(), "alpha_bar 应单调递减"
print("betas:", betas[:3].tolist(), "...", betas[-1].item())


### 1.2 前向采样 `q_sample`(TODO)

实现 `q_sample(x0, t, noise=None)`,返回 $x_t$。  
要求支持 **批量 `t`**(每个样本可有不同的时间步,这是训练时所需)。


In [ ]:
def q_sample(x0: torch.Tensor, t: torch.Tensor, noise: torch.Tensor = None) -> torch.Tensor:
    '''前向加噪。
    Args:
        x0:    (B, ...) 干净样本
        t:     (B,) 时间步索引, dtype=long, 取值范围 [0, T-1]
        noise: 若不提供则自动采样 N(0, I), 形状与 x0 相同
    Returns:
        xt:    (B, ...) 加噪后样本
    '''
    if noise is None:
        noise = torch.randn_like(x0)
    # TODO:
    #   1) 从 alphas_cumprod 中按 t 取值得到 \bar\alpha_t   (形状 (B,))
    #   2) 将其 reshape 到与 x0 可广播的形状, 例如 (B, 1, 1, 1)
    #   3) 按 x_t = sqrt(\bar\alpha_t)*x0 + sqrt(1-\bar\alpha_t)*noise 计算
    raise NotImplementedError


### 1.3 可视化前向过程(给定代码,会调用你写的 `q_sample`)

In [ ]:
from torchvision import datasets, transforms

# 下载 MNIST(若已下载会跳过)
_mnist = datasets.MNIST(
    root="./data", train=True, download=True,
    transform=transforms.ToTensor()
)

# 取一张数字 7 的图,归一化到 [-1, 1]
img, _ = _mnist[0]                       # img: (1, 28, 28) in [0,1]
x0 = img.unsqueeze(0) * 2.0 - 1.0        # (1, 1, 28, 28) in [-1, 1]

ts_to_show = [0, 50, 100, 200, 500, 999]
fig, axes = plt.subplots(1, len(ts_to_show), figsize=(2*len(ts_to_show), 2.2))
for ax, t_val in zip(axes, ts_to_show):
    t = torch.tensor([t_val], dtype=torch.long)
    xt = q_sample(x0, t)                                      # 调用你的实现
    ax.imshow(xt[0, 0].numpy(), cmap="gray", vmin=-1, vmax=1)
    ax.set_title(f"t = {t_val}")
    ax.axis("off")
plt.suptitle("前向扩散过程")
plt.tight_layout(); plt.show()


### 1.4 思考题 ✏️

**Q1.** 观察 $t=999$ 时的图像,几乎看不到原始数字。请用 $\bar\alpha_t$ 的取值范围(画出 $\bar\alpha_t$ vs $t$ 的曲线作为辅助)解释这一现象。

> **回答(请在此 cell 编辑):**
>
> _你的回答……_


In [ ]:
# 辅助:绘制 alpha_bar_t 随 t 的变化
plt.figure(figsize=(5, 3))
plt.plot(alphas_cumprod.numpy())
plt.xlabel("t"); plt.ylabel(r"$\bar\alpha_t$")
plt.title("Cumulative product of alphas")
plt.grid(True); plt.tight_layout(); plt.show()


---
## Task 2:2D 玩具数据集上的 DDPM (30 分)

我们先在 2D 数据上完整实现一遍 DDPM 训练 + 采样,这样 CPU 几分钟就能看到结果,便于调试。


### 2.1 数据准备(已给出)

In [ ]:
from sklearn.datasets import make_swiss_roll

def get_swiss_roll(n_samples=4000, noise=0.5):
    X, _ = make_swiss_roll(n_samples=n_samples, noise=noise, random_state=0)
    X = X[:, [0, 2]]                                 # 取 x, z 两维
    X = X / np.abs(X).max() * 3.0                    # 归一化到 ~[-3, 3]
    return torch.tensor(X, dtype=torch.float32)

data_2d = get_swiss_roll()
print("data shape:", data_2d.shape)

plt.figure(figsize=(4, 4))
plt.scatter(data_2d[:, 0], data_2d[:, 1], s=2)
plt.title("Swiss Roll (2D)"); plt.axis("equal"); plt.show()


### 2.2 时间嵌入 + 小型 MLP(已给出)

我们用一个三层 MLP 作为 $\epsilon_\theta(x_t, t)$,时间步用 **正弦位置编码** 后拼接到输入。


In [ ]:
class SinusoidalTimeEmb(nn.Module):
    '''将时间步 t 映射为一个 dim 维向量, 类似 Transformer 的位置编码。'''
    def __init__(self, dim: int):
        super().__init__()
        self.dim = dim
    def forward(self, t):
        # t: (B,) long
        half = self.dim // 2
        freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device) / half)
        ang = t.float().unsqueeze(1) * freqs.unsqueeze(0)        # (B, half)
        return torch.cat([torch.sin(ang), torch.cos(ang)], dim=1)  # (B, dim)


class ToyMLP(nn.Module):
    def __init__(self, data_dim=2, hidden=64, t_dim=32):
        super().__init__()
        self.time_emb = SinusoidalTimeEmb(t_dim)
        self.net = nn.Sequential(
            nn.Linear(data_dim + t_dim, hidden),
            nn.SiLU(),
            nn.Linear(hidden, hidden),
            nn.SiLU(),
            nn.Linear(hidden, hidden),
            nn.SiLU(),
            nn.Linear(hidden, data_dim),
        )
    def forward(self, x, t):
        te = self.time_emb(t)
        h = torch.cat([x, te], dim=1)
        return self.net(h)


# 看一下参数量
_m = ToyMLP()
print("ToyMLP params:", sum(p.numel() for p in _m.parameters()))


### 2.3 训练损失(TODO)

实现 simplified loss:

$$
\mathcal L = \mathbb E_{x_0,t,\epsilon}\big\|\epsilon - \epsilon_\theta(x_t, t)\big\|^2
$$

注意:每个样本独立采样一个随机 $t \sim \mathcal{U}\{0,\dots,T-1\}$。


In [ ]:
def diffusion_loss(model, x0: torch.Tensor) -> torch.Tensor:
    '''DDPM 的 simplified MSE loss。
    Args:
        model: 接受 (x, t) 返回预测噪声的网络
        x0:    (B, ...) 干净样本
    Returns:
        loss:  标量 tensor
    '''
    B = x0.shape[0]
    # TODO:
    #   1) 为每个样本随机采样 t ~ Uniform{0, ..., T-1}, shape (B,), dtype=long
    #   2) 采样标准高斯噪声 eps, 形状与 x0 相同
    #   3) 用 q_sample 得到 x_t
    #   4) 让 model 预测噪声, 计算 MSE(eps, model(x_t, t))
    raise NotImplementedError


### 2.4 训练循环(已给出,会调用你的 loss)

In [ ]:
def train_toy(model, data, epochs=80, batch_size=256, lr=2e-3):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loader = DataLoader(TensorDataset(data), batch_size=batch_size, shuffle=True)
    history = []
    pbar = tqdm(range(epochs))
    for ep in pbar:
        ep_loss = 0.0
        for (x0,) in loader:
            loss = diffusion_loss(model, x0)
            opt.zero_grad(); loss.backward(); opt.step()
            ep_loss += loss.item() * x0.size(0)
        ep_loss /= len(data)
        history.append(ep_loss)
        pbar.set_postfix(loss=f"{ep_loss:.4f}")
    return history


torch.manual_seed(SEED)
toy_model = ToyMLP()
t0 = time.time()
hist = train_toy(toy_model, data_2d, epochs=80)
print(f"Training time: {time.time()-t0:.1f}s")

plt.figure(figsize=(5,3))
plt.plot(hist); plt.xlabel("epoch"); plt.ylabel("loss")
plt.title("Toy DDPM training loss"); plt.grid(True); plt.show()


### 2.5 反向采样 `ddpm_sample`(TODO)

实现完整的 DDPM 采样:

$$
x_{t-1} = \frac{1}{\sqrt{\alpha_t}}\Big(x_t - \frac{1-\alpha_t}{\sqrt{1-\bar\alpha_t}}\,\epsilon_\theta(x_t,t)\Big) + \sigma_t z
$$

- 取 $\sigma_t^2 = \beta_t$;$t=0$ 时不再加噪($z=0$)。
- 接口为 `ddpm_sample(model, shape) -> tensor of shape`。


In [ ]:
@torch.no_grad()
def ddpm_sample(model, shape) -> torch.Tensor:
    '''从纯噪声出发, 反向去噪生成样本。
    Args:
        model: 训练好的噪声预测网络
        shape: 输出张量形状, 例如 (n, 2) 或 (n, 1, 16, 16)
    Returns:
        x0_hat: shape 形状的样本张量
    '''
    model.eval()
    x = torch.randn(shape)
    # TODO:
    #   按 t = T-1, T-2, ..., 0 依次更新 x:
    #     eps_pred = model(x, t_batch)
    #     mean = (1/sqrt(alpha_t)) * (x - (1-alpha_t)/sqrt(1-alpha_bar_t) * eps_pred)
    #     if t > 0:
    #         x = mean + sqrt(beta_t) * z,  z ~ N(0, I)
    #     else:
    #         x = mean
    raise NotImplementedError


### 2.6 可视化生成结果

In [ ]:
samples = ddpm_sample(toy_model, (2000, 2)).numpy()

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].scatter(data_2d[:, 0], data_2d[:, 1], s=2)
axes[0].set_title("Training data"); axes[0].axis("equal")
axes[1].scatter(samples[:, 0], samples[:, 1], s=2, c="orange")
axes[1].set_title("DDPM samples"); axes[1].axis("equal")
plt.tight_layout(); plt.show()


### 2.7 思考题 ✏️

**Q2.** 把 `T` 从 1000 改成 50(只需重写 `betas` 并重新计算 `alphas`, `alphas_cumprod`,然后重新训练 + 采样),观察:
- 训练 loss 曲线有什么不同?
- 生成样本质量有什么变化?为什么?

> **回答:**
>
> _你的回答……_

> 💡 提示:可以临时复制本任务的代码到下方 cell 做实验,**不要覆盖**原始 `betas`,做完后恢复 T=1000 再进入 Task 3。


In [ ]:
# (可选) 在此处做 T=50 的对照实验
# ...


---
## Task 3:MNIST 上的 DDPM (40 分)

下面将上述完整流程搬到 MNIST。为保证 CPU 训练在一小时内完成,我们:
- 将图像从 28×28 **下采样到 16×16**
- 只使用训练集的 **前 10,000 张**
- 使用一个 ~30 万参数的小型 UNet


### 3.1 数据加载(已给出)

In [ ]:
IMG_SIZE = 16
N_TRAIN  = 10000

tf = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),                     # to [0,1]
    transforms.Lambda(lambda x: x * 2 - 1),    # to [-1,1]
])
full_train = datasets.MNIST("./data", train=True, download=True, transform=tf)
mnist_subset = torch.stack([full_train[i][0] for i in range(N_TRAIN)])  # (N,1,16,16)
print("MNIST subset:", mnist_subset.shape, "min/max:",
      mnist_subset.min().item(), mnist_subset.max().item())

# 看几张样本
fig, axes = plt.subplots(1, 8, figsize=(12, 1.6))
for i, ax in enumerate(axes):
    ax.imshow(mnist_subset[i, 0], cmap="gray", vmin=-1, vmax=1); ax.axis("off")
plt.suptitle("MNIST 16x16 训练样本"); plt.show()


### 3.2 小型 UNet(已给出)

包含:Down → Bottleneck → Up 的对称结构,每个 block 含 GroupNorm + SiLU + Conv,并通过 FiLM 风格将时间嵌入注入。


In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, t_dim):
        super().__init__()
        self.norm1 = nn.GroupNorm(min(8, in_ch), in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.norm2 = nn.GroupNorm(min(8, out_ch), out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.time_proj = nn.Linear(t_dim, out_ch)
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
    def forward(self, x, t_emb):
        h = self.conv1(F.silu(self.norm1(x)))
        h = h + self.time_proj(t_emb).unsqueeze(-1).unsqueeze(-1)
        h = self.conv2(F.silu(self.norm2(h)))
        return h + self.skip(x)


class SmallUNet(nn.Module):
    '''输入 (B,1,16,16), 输出 (B,1,16,16) 预测噪声。'''
    def __init__(self, base_ch=32, t_dim=64):
        super().__init__()
        self.time_emb = nn.Sequential(
            SinusoidalTimeEmb(t_dim),
            nn.Linear(t_dim, t_dim), nn.SiLU(),
            nn.Linear(t_dim, t_dim),
        )
        c1, c2, c3 = base_ch, base_ch*2, base_ch*4

        self.down1 = ConvBlock(1, c1, t_dim)            # 16
        self.down2 = ConvBlock(c1, c2, t_dim)           # 8
        self.mid   = ConvBlock(c2, c3, t_dim)           # 4

        self.up2   = ConvBlock(c3 + c2, c2, t_dim)      # 8
        self.up1   = ConvBlock(c2 + c1, c1, t_dim)      # 16

        self.out   = nn.Conv2d(c1, 1, 1)
        self.pool  = nn.AvgPool2d(2)

    def forward(self, x, t):
        te = self.time_emb(t)
        h1 = self.down1(x, te)                  # (B, c1, 16, 16)
        h2 = self.down2(self.pool(h1), te)      # (B, c2, 8, 8)
        hm = self.mid(self.pool(h2), te)        # (B, c3, 4, 4)
        u2 = F.interpolate(hm, scale_factor=2, mode="nearest")
        u2 = self.up2(torch.cat([u2, h2], dim=1), te)   # (B, c2, 8, 8)
        u1 = F.interpolate(u2, scale_factor=2, mode="nearest")
        u1 = self.up1(torch.cat([u1, h1], dim=1), te)   # (B, c1, 16, 16)
        return self.out(u1)


unet = SmallUNet()
n_params = sum(p.numel() for p in unet.parameters())
print(f"SmallUNet params: {n_params:,}")
# 简单前向测试
_dummy_x = torch.randn(2, 1, 16, 16)
_dummy_t = torch.tensor([10, 100])
assert unet(_dummy_x, _dummy_t).shape == (2, 1, 16, 16)
print("Forward pass OK.")


### 3.3 训练循环(基本已给出,你需要确保 `diffusion_loss` 在图像上也工作)

你在 Task 2 中实现的 `diffusion_loss` 与 `q_sample` 应该 **完全可复用** —— 它们应该对任意张量形状都成立。如果不行,请回去修改使其支持 (B, C, H, W) 形状。


In [ ]:
EPOCHS_MNIST = 15
BATCH_SIZE   = 128

def train_mnist(model, data, epochs=EPOCHS_MNIST, batch_size=BATCH_SIZE, lr=2e-4):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loader = DataLoader(TensorDataset(data), batch_size=batch_size, shuffle=True)
    hist = []
    for ep in range(epochs):
        model.train()
        ep_loss, n = 0.0, 0
        pbar = tqdm(loader, desc=f"epoch {ep+1}/{epochs}", leave=False)
        for (x0,) in pbar:
            loss = diffusion_loss(model, x0)
            opt.zero_grad(); loss.backward(); opt.step()
            ep_loss += loss.item() * x0.size(0); n += x0.size(0)
            pbar.set_postfix(loss=f"{loss.item():.4f}")
        hist.append(ep_loss / n)
        print(f"  epoch {ep+1:2d} | loss = {hist[-1]:.4f}")
    return hist


torch.manual_seed(SEED)
unet = SmallUNet()
t0 = time.time()
hist_unet = train_mnist(unet, mnist_subset)
print(f"Total training time: {(time.time()-t0)/60:.1f} min")

plt.figure(figsize=(5,3))
plt.plot(hist_unet, marker="o")
plt.xlabel("epoch"); plt.ylabel("loss")
plt.title("MNIST DDPM loss"); plt.grid(True); plt.show()


### 3.4 生成 64 张 MNIST 图像

直接复用你在 Task 2 写的 `ddpm_sample`,把 shape 改成 `(64, 1, 16, 16)` 即可。


In [ ]:
t0 = time.time()
gen = ddpm_sample(unet, (64, 1, 16, 16))
print(f"Sampling 64 images took {time.time()-t0:.1f}s")

gen = gen.clamp(-1, 1)
fig, axes = plt.subplots(8, 8, figsize=(8, 8))
for i, ax in enumerate(axes.flatten()):
    ax.imshow(gen[i, 0].cpu().numpy(), cmap="gray", vmin=-1, vmax=1)
    ax.axis("off")
plt.suptitle("DDPM Samples on MNIST (16x16)"); plt.tight_layout(); plt.show()


### 3.5 思考题 ✏️

**Q3.** 观察你生成的 64 张图像:
- 大约有多少张可以辨认为某个数字?
- 哪些数字最容易/最难生成?有什么共同特征?
- 如果给你两倍的训练时间,你会如何调整超参数(`T`, 学习率, epoch, UNet 通道数)?为什么?

> **回答:**
>
> _你的回答……_


---
## Task 4(加分,10 分):DDIM 加速采样

DDPM 默认需要 $T=1000$ 步才能采样一张图,DDIM (Song et al. 2021) 通过确定性的非马尔可夫过程,可以**只用 20~50 步**就达到接近的质量。

DDIM 的更新公式($\eta=0$,确定性版本):

$$
x_{t_{i-1}} = \sqrt{\bar\alpha_{t_{i-1}}}\,\hat x_0 + \sqrt{1-\bar\alpha_{t_{i-1}}}\,\epsilon_\theta(x_{t_i}, t_i)
$$

其中

$$
\hat x_0 = \frac{x_{t_i} - \sqrt{1-\bar\alpha_{t_i}}\,\epsilon_\theta(x_{t_i}, t_i)}{\sqrt{\bar\alpha_{t_i}}}
$$

时间步序列 $t_S > t_{S-1} > \dots > t_0 = 0$ 可在 $[0, T)$ 中均匀子采样得到。


In [ ]:
@torch.no_grad()
def ddim_sample(model, shape, n_steps: int = 20) -> torch.Tensor:
    '''DDIM 确定性采样 (eta=0)。
    Args:
        model:    噪声预测网络
        shape:    输出形状
        n_steps:  采样步数 (远小于 T)
    '''
    model.eval()
    # 构造子序列 t_S, ..., t_0
    step_idx = torch.linspace(T - 1, 0, n_steps + 1).long()  # (n_steps+1,)
    x = torch.randn(shape)
    # TODO:
    #   for i in range(n_steps):
    #       t_curr = step_idx[i]; t_prev = step_idx[i+1]
    #       eps = model(x, t_curr_batch)
    #       x0_hat = (x - sqrt(1-alpha_bar_curr) * eps) / sqrt(alpha_bar_curr)
    #       x = sqrt(alpha_bar_prev) * x0_hat + sqrt(1-alpha_bar_prev) * eps
    raise NotImplementedError


In [ ]:
# 对比 DDPM vs DDIM 的耗时与质量
t0 = time.time()
gen_ddpm = ddpm_sample(unet, (16, 1, 16, 16))
t_ddpm = time.time() - t0

t0 = time.time()
gen_ddim = ddim_sample(unet, (16, 1, 16, 16), n_steps=20)
t_ddim = time.time() - t0

print(f"DDPM (T={T}) sampling 16 imgs: {t_ddpm:.1f}s")
print(f"DDIM (S=20)  sampling 16 imgs: {t_ddim:.1f}s")

fig, axes = plt.subplots(2, 16, figsize=(16, 2.4))
for i in range(16):
    axes[0, i].imshow(gen_ddpm[i, 0].clamp(-1,1), cmap="gray", vmin=-1, vmax=1); axes[0, i].axis("off")
    axes[1, i].imshow(gen_ddim[i, 0].clamp(-1,1), cmap="gray", vmin=-1, vmax=1); axes[1, i].axis("off")
axes[0, 0].set_ylabel("DDPM"); axes[1, 0].set_ylabel("DDIM")
plt.suptitle(f"Top: DDPM ({T} steps, {t_ddpm:.1f}s)  |  Bottom: DDIM (20 steps, {t_ddim:.1f}s)")
plt.show()


### 4.1 思考题 ✏️

**Q4.** DDPM 每一步都依赖于一个**随机** 噪声项 $\sigma_t z$,而 DDIM ($\eta=0$) 是完全确定性的。请尝试回答:
1. 为什么 DDIM 可以"跳步"而 DDPM 通常不行?(提示:从马尔可夫性质角度思考)
2. DDIM 确定性采样有什么额外好处?(提示:同一个噪声 → 同一张图)

> **回答:**
>
> _你的回答……_


---
## 完成清单 ✅

- [ ] Task 1: `make_beta_schedule`, `alphas`, `alphas_cumprod`, `q_sample` 已实现
- [ ] Task 1: 前向过程可视化、Q1 回答完成
- [ ] Task 2: `diffusion_loss`, `ddpm_sample` 已实现
- [ ] Task 2: 2D 训练曲线、生成散点图、Q2 回答完成
- [ ] Task 3: MNIST 训练 loss 曲线、64 张生成图、Q3 回答完成
- [ ] (可选)Task 4: `ddim_sample` 实现、DDPM vs DDIM 对比、Q4 回答完成
- [ ] 已 `Restart & Run All`,所有 cell 顺序运行通过
- [ ] 文件已重命名为 `学号_姓名_diffusion_lab.ipynb`

提交至课程平台即可。Good luck!🎉
